In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# 1. Inicializar sesión de Spark
spark = SparkSession.builder.appName("SilverToGoldSales").getOrCreate()

# 2. Definir rutas de las capas en Unity Catalog
ruta_silver = "/Volumes/workspace/default/csvfiles/silver/sales_clean"
ruta_destino_gold_clientes = "/Volumes/workspace/default/csvfiles/gold/sales_by_customer"
ruta_destino_gold_paises = "/Volumes/workspace/default/csvfiles/gold/sales_by_country"

print("--- INICIO DE PROCESAMIENTO SILVER A GOLD ---")

# 3. Leer los datos limpios de la capa Silver (Formato Delta)
df_silver = spark.read.format("delta").load(ruta_silver)

# 4. Aplicar Filtro de Negocio (Regla Gold)
# Para reportes financieros/comerciales, solo tomamos transacciones consolidadas y NO canceladas
df_ventas_efectivas = df_silver.filter((F.col("Es_Venta") == True) & (F.col("Cancelada") == False))

# ------------------------------------------------------------------------
# AGREGACIÓN 1: Métricas de Negocio por Cliente (Empresas Reales)
# ------------------------------------------------------------------------
df_gold_customer = df_ventas_efectivas.groupBy("Cliente", "Pais") \
    .agg(
        F.count("ID_Venta").alias("Total_Ordenes"),
        F.sum("Cantidad").alias("Unidades_Vendidas"),
        F.round(F.sum("Total"), 2).alias("Ingresos_Totales"),
        F.round(F.avg("Total"), 2).alias("Ticket_Promedio") \
    ) \
    .orderBy(F.desc("Ingresos_Totales"))

# ------------------------------------------------------------------------
# AGREGACIÓN 2: Métricas de Rendimiento por País y Categoría
# ------------------------------------------------------------------------
df_gold_country = df_ventas_efectivas.groupBy("Pais", "Producto_Categoria") \
    .agg(
        F.sum("Cantidad").alias("Total_Unidades"),
        F.round(F.sum("Total"), 2).alias("Monto_Total")
    ) \
    .orderBy("Pais", F.desc("Monto_Total"))

# ------------------------------------------------------------------------
# 5. Almacenamiento en la Capa Gold (Formato Delta)
# Como son tablas agregadas (pequeñas y resumidas), usamos overwrite para actualizar el reporte completo
# ------------------------------------------------------------------------
df_gold_customer.write.format("delta") \
    .mode("overwrite") \
    .save(ruta_destino_gold_clientes)

df_gold_country.write.format("delta") \
    .mode("overwrite") \
    .save(ruta_destino_gold_paises)

print("¡Capa Gold generada exitosamente!")
print(f"Reporte Clientes guardado en: {ruta_destino_gold_clientes}")
print(f"Reporte Países guardado en: {ruta_destino_gold_paises}")

# 6. Mostrar una vista previa del resultado de negocio
print("\n--- VISTA PREVIA: TOP EMPRESAS POR INGRESOS ---")
display(df_gold_customer.limit(5))
